# RAG with Ollama, LangChain, and FAISS 🧠 (Optimized and Updated Version)
# This script demonstrates an optimized Retrieval-Augmented Generation (RAG) pipeline
# with improved performance, logging, and progress tracking.

In [1]:
# ==============================================================================
# CELL 1: Setup and Installations
# ==============================================================================
import os
import logging
import re
from tqdm import tqdm
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --- Enhanced Logging Setup ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

logging.info("--- Initializing AI Medical Assistant with Local Ollama Models ---")

2025-08-26 16:29:31,022 - INFO - --- Initializing AI Medical Assistant with Local Ollama Models ---


In [2]:
# ==============================================================================
# CELL 2: Prepare the Knowledge Base from a folder of PDFs
# ==============================================================================
knowledge_base_dir = "knowledge_base"
if not os.path.exists(knowledge_base_dir):
    os.makedirs(knowledge_base_dir)
    logging.warning(f"Created a folder '{knowledge_base_dir}'. Please add your PDF files to it and rerun.")
    # exit() # Commented out to allow notebook to run fully in all environments

all_chunks = []
pdf_files = [f for f in os.listdir(knowledge_base_dir) if f.endswith('.pdf')]

if not pdf_files:
    logging.error(f"No PDF files found in '{knowledge_base_dir}'. Please add PDFs.")
    # exit()
else:
    logging.info(f"[Step 1] Loading and splitting documents from '{knowledge_base_dir}'...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    
    for pdf_file in tqdm(pdf_files, desc="Processing PDFs"):
        pdf_path = os.path.join(knowledge_base_dir, pdf_file)
        try:
            loader = PyPDFLoader(pdf_path)
            docs = loader.load_and_split(text_splitter=text_splitter)
            all_chunks.extend(docs)
            logging.info(f"Loaded and split {len(docs)} chunks from {pdf_file}")
        except Exception as e:
            logging.error(f"Failed to process {pdf_file}. Error: {e}")

    if not all_chunks:
        logging.error("Could not extract any text from the PDF files. Exiting.")
        # exit()
    else:
        logging.info(f"Successfully split documents into a total of {len(all_chunks)} chunks.")

2025-08-26 16:31:33,773 - INFO - [Step 1] Loading and splitting documents from 'knowledge_base'...
Processing PDFs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14/14 [00:31<00:00,  2.24s/it]
2025-08-26 16:32:05,159 - INFO - Successfully split documents into a total of 451 chunks.


In [ ]:
# ==============================================================================
# CELL 3: Create or Load Vector Store (High-Quality Embeddings in Batches)
# ==============================================================================
vector_store_path = "faiss_index_ollama_hq" # Using a new path for the HQ index
faiss_index_file = os.path.join(vector_store_path, "index.faiss")

# UPGRADED MODEL: 'nomic-embed-text' for higher quality context retrieval.
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vector_store = None
if os.path.exists(faiss_index_file):
    logging.info(f"[Step 2] Loading existing vector store from '{vector_store_path}'...")
    try:
        vector_store = FAISS.load_local(
            vector_store_path,
            embeddings,
            allow_dangerous_deserialization=True
        )
        logging.info("Vector store loaded successfully!")
    except Exception as e:
        logging.error(f"Failed to load vector store: {e}. Will create a new one.")

if not vector_store and all_chunks:
    logging.info("[Step 2] Creating new high-quality vector store in batches...")
    logging.info("This will take a significant amount of time (est. 1-1.5 hours).")
    try:
        # Define the size of each batch to process.
        BATCH_SIZE = 32
        
        # The first batch creates the vector store.
        logging.info(f"--> Processing initial batch of {min(BATCH_SIZE, len(all_chunks))} chunks...")
        first_batch = all_chunks[:BATCH_SIZE]
        vector_store = FAISS.from_documents(documents=first_batch, embedding=embeddings)
        
        # Subsequent batches are added to the existing store.
        num_batches = (len(all_chunks) - 1) // BATCH_SIZE + 1
        for i in range(1, num_batches):
            start_index = i * BATCH_SIZE
            end_index = start_index + BATCH_SIZE
            next_batch = all_chunks[start_index:end_index]
            
            logging.info(f"--> Processing batch {i+1}/{num_batches} ({len(next_batch)} chunks)...")
            vector_store.add_documents(documents=next_batch)

        # Save the completed vector store to disk.
        logging.info("All batches processed. Saving vector store to disk...")
        vector_store.save_local(vector_store_path)
        logging.info(f"Vector store created and saved to '{vector_store_path}'!")

    except Exception as e:
        logging.error(f"Failed to create vector store: {e}")
        # exit()

2025-08-26 16:34:17,881 - INFO - [Step 2] Creating new high-quality vector store in batches...
2025-08-26 16:34:17,884 - INFO - This will take a significant amount of time (est. 1-1.5 hours).
2025-08-26 16:34:17,887 - INFO - --> Processing initial batch of 32 chunks...
2025-08-26 16:47:52,666 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-08-26 16:47:52,693 - INFO - Loading faiss.
2025-08-26 16:47:52,768 - INFO - Successfully loaded faiss.
2025-08-26 16:47:52,825 - INFO - --> Processing batch 2/15 (32 chunks)...


In [ ]:
# ==============================================================================
# CELL 4: Build the Improved RAG and Non-RAG Chains (High-Quality Model)
# ==============================================================================

# --- 1. Define Retriever and Model ---
retriever = vector_store.as_retriever()
# UPGRADED MODEL: 'llama3:8b' for superior reasoning and question generation.
model = ChatOllama(model="llama3.1:8b")

# --- 2. Create the New, Simplified Prompt Template ---
# This prompt is much more direct. It clearly defines the role, the context,
# the task, and the desired output format.
template = """
You are an AI medical assistant. Your task is to help a patient organize their thoughts before a doctor's visit.
Based on the provided medical context and the patient's symptom, generate exactly 5 questions to ask the patient about their medical history. These questions should help the patient reflect and structure their information effectively for their doctor.

MEDICAL CONTEXT:
{context}

PATIENT'S SYMPTOM:
{symptom}

YOUR 5 QUESTIONS FOR THE PATIENT:
"""

prompt = ChatPromptTemplate.from_template(template)

# --- 3. Build the RAG Chain ---
rag_chain = (
    {
        "context": retriever,
        "symptom": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

# --- 4. Build the Non-RAG Chain for comparison ---
# This chain uses the same prompt but passes an empty string for context.
non_rag_chain = (
    {
        "context": lambda x: "", # Always provide empty context
        "symptom": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

logging.info("[Step 3] AI Assistant RAG and Non-RAG chains are ready.")

In [ ]:
# ==============================================================================
# CELL 5: Ask Questions and Get Actionable Results
# ==============================================================================

def get_questions_for_symptom(symptom):
    """Invokes both RAG and non-RAG chains for a given symptom and prints the results."""
    if not vector_store:
        logging.error("Vector store is not available. Cannot process symptom.")
        return
        
    logging.info(f"Processing symptom: '{symptom}'")
    print(f"\n{'='*80}")
    print(f"PATIENT'S SYMPTOM: {symptom}")
    print(f"{'-'*80}")

    # --- Non-RAG Invocation ---
    print("\n🤖 Generating questions WITHOUT RAG (from model's general knowledge)...")
    print("(This may take 60-90 minutes)")
    try:
        non_rag_response = non_rag_chain.invoke(symptom)
        questions = re.findall(r'\d+\.\s(.*?)(?=\n\d+\.|\Z)', non_rag_response, re.DOTALL)
        if questions:
            for i, q in enumerate(questions, 1):
                print(f"{i}. {q.strip()}")
        else:
            print("Could not parse questions. Raw response:\n", non_rag_response)
    except Exception as e:
        logging.error(f"An error occurred during Non-RAG chain invocation: {e}")

    print(f"\n{'-'*40}\n")

    # --- RAG Invocation ---
    print("📚 Generating questions WITH RAG (guided by your documents)...")
    print("(This may take 60-90 minutes)")
    try:
        rag_response = rag_chain.invoke(symptom)
        questions = re.findall(r'\d+\.\s(.*?)(?=\n\d+\.|\Z)', rag_response, re.DOTALL)
        if questions:
            for i, q in enumerate(questions, 1):
                print(f"{i}. {q.strip()}")
        else:
            print("Could not parse questions. Raw response:\n", rag_response)
    except Exception as e:
        logging.error(f"An error occurred during RAG chain invocation: {e}")

    print(f"\n{'='*80}")

# --- Start the AI Assistant Session ---
print("\n--- AI Medical Assistant Session ---")

# Test with the same symptoms from your original notebook
get_questions_for_symptom("My right knee hurts.")
get_questions_for_symptom("I am feeling tired all the time.")
get_questions_for_symptom("I was crouched and when I rised up, I got dizzy and my vision went black for about 15 seconds.")

logging.info("--- Session finished ---")